# Neural Style Transfer

Neural Style Transfer (NST) is a technique which combines two images (**content image** for the object of the image and **style image** from which only the style is extracted (into a third target image.

The technique was first conceptualized by Leon A. Gatys, Alexander S. Ecker, and Matthias Bethge in the paper [A Neural Algorithm of Artistic Style](https://arxiv.org/abs/1508.06576).

## Basic Concept

Take two losses and minimize them as much as possible.

The first loss describes the distance in content between the content image and the target image; which the second loss describes the distance in style between the style image and the style of the input image.


![Content and Style Image](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*X48OwUuUIvgAGoo8ydGI0Q.png)

*The content image (left) and style image (right).*

![Final Result](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*MPUfyVdiQ_CBsTqS1RirYg.png)

*The final result combining the content and style image.*



Let's start with importing  all relevant libraries.
- TensorFlow for deep learning,
- NumPy for numerical operations,
- Matplotlib for data visualisation

In [ ]:
import os
from matplotlib import gridspec
import matplotlib.pylab as plt
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

print("TF Version: ", tf.__version__)
print("TF Hub version: ", hub.__version__)
print("GPU available: ", tf.config.list_physical_devices('GPU'))

### Define image loading and visualization functions

In [ ]:
def crop_center(image):
  """Returns a cropped square image."""
  shape = image.shape
  new_shape = min(shape[1], shape[2])
  offset_y = max(shape[1] - shape[2], 0) // 2
  offset_x = max(shape[2] - shape[1], 0) // 2
  image = tf.image.crop_to_bounding_box(
      image, offset_y, offset_x, new_shape, new_shape)
  return image

def load_image(image_url, image_size=(256, 256), from_local=False, preserve_aspect_ratio=True):
  """Loads and preprocesses images."""
  # Cache image file locally.

  if (from_local==False):
    image_path = tf.keras.utils.get_file(os.path.basename(image_url)[-128:], image_url)
  else:
    image_path=image_url
  # Load and convert to float32 numpy array, add batch dimension, and normalize to range [0, 1].
  img = tf.io.decode_image(
      tf.io.read_file(image_path),
      channels=3, dtype=tf.float32)[tf.newaxis, ...]
  img = crop_center(img)
  img = tf.image.resize(img, image_size, preserve_aspect_ratio=True)
  return img

def show_n(images, titles=('',)):
  n = len(images)
  image_sizes = [image.shape[1] for image in images]
  w = (image_sizes[0] * 6) // 320
  plt.figure(figsize=(w * n, w))
  gs = gridspec.GridSpec(1, n, width_ratios=image_sizes)
  for i in range(n):
    plt.subplot(gs[i])
    plt.imshow(images[i][0], aspect='equal')
    plt.axis('off')
    plt.title(titles[i] if len(titles) > i else '')
  plt.show()


Let's upload two images to play with.

- **Content image**: the photo or object you want to keep.
- **Style image**: the painting, drawing, texture, or pattern whose style you want to copy.

Using uploaded images avoids broken links and `HTTP Error 403: Forbidden` errors from external websites.

In [ ]:
# @title Upload your content image and style image { display-mode: "form" }

from google.colab import files

output_image_size = 384  # @param {type:"integer"}
content_img_size = (output_image_size, output_image_size)
style_img_size = (256, 256)  # Recommended size for the style image.

print("Step 1: Upload your CONTENT image. This is the photo/object you want to keep.")
content_upload = files.upload()
content_filename = next(iter(content_upload))

print("Step 2: Upload your STYLE image. This is the painting/drawing/texture style you want to apply.")
style_upload = files.upload()
style_filename = next(iter(style_upload))

# Load and resize the uploaded images.
content_image = load_image(content_filename, content_img_size, from_local=True)
style_image = load_image(style_filename, style_img_size, from_local=True)

show_n([content_image, style_image], ['Content image', 'Style image'])


## Import TF Hub module

In [ ]:
# Load TF Hub module.

hub_handle = 'https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2'
hub_module = hub.load(hub_handle)

## Demonstrate image stylization

In [ ]:
outputs = hub_module(tf.constant(content_image), tf.constant(style_image))
stylized_image = outputs[0]

In [ ]:
# Visualize input images and the generated stylized image.

show_n([content_image, style_image, stylized_image], titles=['Original content image', 'Style image', 'Stylized image'])

## Try it again with your own images

Run the next cell when you want to upload a new pair of images and generate another stylized result.

In [ ]:
# @title Upload a new pair of images and stylize them { display-mode: "form" }

from google.colab import files

output_image_size = 384  # @param {type:"integer"}
content_img_size = (output_image_size, output_image_size)
style_img_size = (256, 256)

print("Upload a new CONTENT image.")
content_upload = files.upload()
content_filename = next(iter(content_upload))

print("Upload a new STYLE image.")
style_upload = files.upload()
style_filename = next(iter(style_upload))

content_image = load_image(content_filename, content_img_size, from_local=True)
style_image = load_image(style_filename, style_img_size, from_local=True)

stylized_image = hub_module(tf.constant(content_image),
                            tf.constant(style_image))[0]

show_n([content_image, style_image, stylized_image],
       titles=['Original content image', 'Style image', 'Stylized image'])


### Image upload tips

For the content image, choose a clear photo with one main subject. For the style image, choose a painting, drawing, texture, or pattern with strong colors or brush strokes. JPG and PNG files work best.

# **Task**

Create your own neural style transfer result by uploading two images.

**Step 1:** Upload a **content image**. This is the image whose objects and layout should stay the same. Examples: a photo of a bridge, a pet, a face, a toy, or a landscape.

**Step 2:** Upload a **style image**. This is the image whose artistic style should be copied. Examples: a painting, a sketch, a watercolor image, a cartoon, or a colorful texture.

**Step 3:** Run the stylization cell and compare the three images: content image, style image, and stylized output.

Questions:
- What parts of the content image were preserved?
- What colors or patterns came from the style image?
- What happens if you try a very simple style image versus a very detailed style image?


In [ ]:
# @title Student practice: upload your own images { display-mode: "form" }

from google.colab import files

output_image_size = 384  # @param {type:"integer"}
content_img_size = (output_image_size, output_image_size)
style_img_size = (256, 256)

print("Upload your CONTENT image first.")
content_upload = files.upload()
content_filename = next(iter(content_upload))

print("Now upload your STYLE image.")
style_upload = files.upload()
style_filename = next(iter(style_upload))

content_image = load_image(content_filename, content_img_size, from_local=True)
style_image = load_image(style_filename, style_img_size, from_local=True)

stylized_image = hub_module(tf.constant(content_image),
                            tf.constant(style_image))[0]

show_n([content_image, style_image, stylized_image],
       titles=['Original content image', 'Style image', 'Stylized image'])
